# GIL en Python: Global Interpreter Lock
- **¿Qué es el GIL?** El Global Interpreter Lock (GIL) de Python es un cerrojo global que asegura que solo un thread (hilo) ejecuta bytecode de Python a la vez, incluso en sistemas multi-core. Es un mutex a nivel de intérprete que limita la ejecución concurrente de threads en Python.
- **Impacto** en la ejecución paralela: Debido al GIL, los hilos en Python no corren en paralelo en tareas CPU-bound (intensivas en CPU). Aunque se lance más de un hilo, únicamente uno estará ejecutando código Python a cualquier instante, los demás esperan el lock. Esto elimina las ventajas de múltiples núcleos para threads Python en cálculos pesados. Ejemplo: sumar números con 2 threads Python suele tardar casi lo mismo que con 1 thread, porque los threads se turnan bajo el GIL.
- **Threads vs Procesos en Python**: La librería threading permite concurrencia pero no paralelismo real en CPU-bound. En contraste, el módulo multiprocessing crea procesos separados (cada uno con su propio intérprete y GIL), logrando aprovechar varios núcleos en paralelo. Para tareas CPU-bound en Python, se recomienda usar procesos en vez de threads. Los threads Python sí pueden mejorar rendimiento en tareas I/O-bound (esperas de entrada/salida), ya que las operaciones bloqueantes liberan el GIL.


# GIL en Python — Concurrencia vs Paralelismo (Cuaderno Colab)

**Objetivo:** Medir y comprender el efecto del **Global Interpreter Lock (GIL)** en Python con experimentos reproducibles.

**Mensajes clave:**
- **Concurrencia** permite estructurar/solapar tareas; **paralelismo** implica ejecución **simultánea real** (múltiples núcleos).
- En **CPython**, el **GIL** garantiza que **solo un hilo** ejecute bytecode Python a la vez.
- **Threads + CPU-bound** ⇒ concurrencia **sin** paralelismo real (no hay speedup).
- **Threads + I/O-bound** ⇒ mejoras por solapamiento de esperas (liberan GIL).
- **Multiprocessing** ⇒ paralelismo real (cada proceso tiene su propio intérprete/GIL).


## 0) Información del entorno

In [1]:
import os, sys, platform, multiprocessing as mp
print("Python:", sys.version)
print("Intérprete:", platform.python_implementation())
print("CPU count (os.cpu_count):", os.cpu_count())
print("CPU count (multiprocessing):", mp.cpu_count())

Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Intérprete: CPython
CPU count (os.cpu_count): 2
CPU count (multiprocessing): 2


## 1) Parámetros de los experimentos
Ajustá estos parámetros si querés acelerar/ralentizar las pruebas.


In [2]:
N_CPU = 10_000_00          # tamaño del cómputo para CPU-bound (1e6 ~ 2e6 razonable en Colab)
N_IO  = 40                  # cantidad de operaciones I/O simuladas
SLEEP = 0.1                 # latencia simulada de I/O (segundos)
REPEAT = 1                  # repeticiones para promediar (ajustá a >1 si querés mayor robustez)

print(dict(N_CPU=N_CPU, N_IO=N_IO, SLEEP=SLEEP, REPEAT=REPEAT))

{'N_CPU': 1000000, 'N_IO': 40, 'SLEEP': 0.1, 'REPEAT': 1}


## 2) Helpers para bench y tareas

In [5]:
import time
from statistics import mean

def bench(fn, *args, repeat=REPEAT, **kwargs):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args, **kwargs)
        times.append(time.perf_counter() - t0)
    return mean(times)

def cpu_task(n: int) -> int:
    s = 0
    for i in range(n):
        s += i*i
    return s

def io_task(lat: float):
    time.sleep(lat)
    return 1

## 3) Baseline CPU-bound (secuencial)

Medimos el tiempo en secuencial para **dos tareas CPU-bound** consecutivas.


In [6]:
t0 = time.perf_counter()
_ = cpu_task(N_CPU); _ = cpu_task(N_CPU)
print("Secuencial 2x (CPU-bound):", time.perf_counter() - t0, "seg")

Secuencial 2x (CPU-bound): 0.10113196899999366 seg


## 4) CPU-bound con **threads** (Concurrencia **sin** paralelismo real en CPython)

Esperamos tiempos cercanos al secuencial. Los hilos se **turnan** por el GIL.


In [7]:
import threading, time

def run_cpu_threads():
    th1 = threading.Thread(target=cpu_task, args=(N_CPU,))
    th2 = threading.Thread(target=cpu_task, args=(N_CPU,))
    t0 = time.perf_counter()
    th1.start(); th2.start()
    th1.join(); th2.join()
    return time.perf_counter() - t0

print("CPU-bound con 2 threads:", run_cpu_threads(), "seg")
print("(Concurrencia SÍ, paralelismo real NO — GIL)")

CPU-bound con 2 threads: 0.10677537700007633 seg
(Concurrencia SÍ, paralelismo real NO — GIL)


## 5) CPU-bound con **processes** (Paralelismo real)

Cada proceso tiene su **propio intérprete** y su **propio GIL**. Si hay núcleos suficientes, veremos **speedup**.


In [8]:
from concurrent.futures import ProcessPoolExecutor

def cpu_task_proc(n: int) -> int:
    # función separada (picklable) para procesos
    s = 0
    for i in range(n):
        s += i*i
    return s

def run_cpu_processes():
    t0 = time.perf_counter()
    with ProcessPoolExecutor() as ex:
        f1 = ex.submit(cpu_task_proc, N_CPU)
        f2 = ex.submit(cpu_task_proc, N_CPU)
        _ = f1.result() + f2.result()
    return time.perf_counter() - t0

print("CPU-bound con 2 procesos:", run_cpu_processes(), "seg")
print("(Concurrencia SÍ, paralelismo real SÍ — depende de núcleos/planificador)")

CPU-bound con 2 procesos: 0.10054896799999824 seg
(Concurrencia SÍ, paralelismo real SÍ — depende de núcleos/planificador)


## 6) I/O-bound baseline (secuencial)

Simulamos I/O con `sleep` que **libera el GIL**. En secuencial, el tiempo ≈ `N_IO * SLEEP`.


In [9]:
t0 = time.perf_counter()
for _ in range(N_IO):
    io_task(SLEEP)
print("I/O secuencial:", time.perf_counter() - t0, "seg")

I/O secuencial: 4.006674652000015 seg


## 7) I/O-bound con **threads** (Concurrencia útil)

Los *threads* pueden **solapar esperas**; mejora el tiempo total aunque no haya paralelismo real.


In [10]:
import threading

def run_io_threads(n=N_IO):
    ths = [threading.Thread(target=io_task, args=(SLEEP,)) for _ in range(n)]
    t0 = time.perf_counter()
    for t in ths: t.start()
    for t in ths: t.join()
    return time.perf_counter() - t0

print("I/O con threads:", run_io_threads(), "seg")
print("(Concurrencia SÍ; paralelismo NO necesario para mejorar)")

I/O con threads: 0.10575290299993867 seg
(Concurrencia SÍ; paralelismo NO necesario para mejorar)


## 8) (Opcional) I/O-bound con **asyncio** (Concurrencia cooperativa)

`async/await` **no** es paralelismo; permite solapar esperas con un **event loop**.


In [12]:
import asyncio, time

async def tarea(i):
    await asyncio.sleep(SLEEP)
    return i

async def main():
    t0 = time.perf_counter()
    res = await asyncio.gather(*(tarea(i) for i in range(N_IO)))
    print("I/O con asyncio:", time.perf_counter() - t0, "seg", "| tareas:", len(res))

try:
    # Si ya hay un loop (notebook), usá await
    asyncio.get_running_loop()
    await main()
except RuntimeError:
    # Si no hay loop (script), usá asyncio.run()
    asyncio.run(main())


I/O con asyncio: 0.10157236800000646 seg | tareas: 40


## 9) Discusión

- **Threads + CPU-bound (CPython):** concurrencia **sin** paralelismo ⇒ tiempos cercanos al secuencial (GIL).

- **Processes + CPU-bound:** puede haber **paralelismo real** ⇒ speedup si hay núcleos.

- **Threads/async + I/O-bound:** mejora por **solapamiento** de esperas (liberan GIL / cooperan con el event loop).

- **Sistema Operativo**: el **planificador** y # de **núcleos** determinan el grado de paralelismo real.

- **Regla práctica**: CPU-bound ⇒ `multiprocessing` o librerías que liberen GIL. I/O-bound ⇒ `threading` o `asyncio`.


## 10) Ejercicios

1. **Ajustar tamaños**: modifica `N_CPU`, `N_IO`, `SLEEP` y `REPEAT` y vuelve a medir.

2. **Escalar procesos**: usa `ProcessPoolExecutor(max_workers=4)` y compara con 1, 2, 4 procesos.

3. **Comparar I/O**: implementa una versión con `ThreadPoolExecutor` y compárala con `asyncio`.

4. **Bonus**: usa `numpy` para una operación vectorizada y observa cómo **libera el GIL** en secciones nativas.
